### Task 7: Mini Use Case

In [3]:
# Perform sentiment classification (positive vs negative reviews)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import os

In [73]:
# CSV files 

file_path = "amazon_reviews.csv"

if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
    df = pd.read_csv(file_path,encoding="utf-8", on_bad_lines="skip")
    print("✅ CSV Loaded Successfully")
    print(df.head())
else:
    print("⚠️ File not found or empty")
    df = pd.DataFrame()

✅ CSV Loaded Successfully
         reviewer_name                                        review_text
0                   KT  This book breaks down the often-intimidating w...
1                laura            Easy to read and extremely informative!
2      Courtney Miller  My team references this book often when explai...
3  Chaminda Ranasinghe  This book gives a simple and methodical backgr...
4                mnowa  Presents mathematical concepts in an approacha...


In [74]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

def label_sentiment(text):
    score = sia.polarity_scores(text)["compound"]
    
    if score >= 0.05:
        return 1
    elif score <= -0.05:
        return 0
    else:
        return None
df["review_text"] = df["review_text"].fillna("").astype(str)

df["label"] = df["review_text"].apply(label_sentiment)
df = df.dropna(subset=["label"])

df["review_text"] = df["review_text"].fillna("").astype(str)




[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [25]:
df_pos = df[df["label"] == 1]
df_neg = df[df["label"] == 0]

df = pd.concat([
    df_pos.sample(len(df_neg), random_state=42),
    df_neg
])

In [87]:
#Navie Bayes Model
from sklearn.naive_bayes import MultinomialNB
df["review_text"] = df["review_text"].fillna("").astype(str)

X = df["review_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),      # captures phrases like "not good"
    stop_words='english',
    max_features=5000
)


X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Accuracy: 0.9130434782608695
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         2
         1.0       0.91      1.00      0.95        21

    accuracy                           0.91        23
   macro avg       0.46      0.50      0.48        23
weighted avg       0.83      0.91      0.87        23



c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [83]:
def predict_sentiment(text_list):
    X_new = vectorizer.transform(text_list)
    preds = model.predict(X_new)
    
    return ["Positive" if p == 1 else "Negative" for p in preds]

# Test
reviews = [
    "This product is excellent",
    "Terrible experience, I hate it"
]

print(predict_sentiment(reviews))

['Positive', 'Positive']


In [84]:
print(y.value_counts())

label
1.0    101
0.0     12
Name: count, dtype: int64


In [85]:
print(y.value_counts())

label
1.0    101
0.0     12
Name: count, dtype: int64


In [76]:
#Logostic regression model & SVM Model
X_train, X_test, y_train, y_test = train_test_split(
    df["review_text"], df["label"], test_size=0.3, random_state=42
)

In [65]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),   # include phrases
    stop_words='english',
    max_features=5000
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [60]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X_train_tfidf, y_train)

c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC()

In [66]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [67]:
y_pred = model.predict(X_test_tfidf)

In [68]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9411764705882353

Classification Report:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         2
         1.0       0.94      1.00      0.97        32

    accuracy                           0.94        34
   macro avg       0.47      0.50      0.48        34
weighted avg       0.89      0.94      0.91        34



c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\D_Drive\FullStackGenai\fsgenai\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [69]:
new_reviews = [
    "This product is excellent",
    "Terrible experience, I hate it"
]

new_tfidf = vectorizer.transform(new_reviews)
predictions = model.predict(new_tfidf)

for review, pred in zip(new_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"{review} --> {sentiment}")

This product is excellent --> Positive
Terrible experience, I hate it --> Positive


In [70]:
print(model.predict(vectorizer.transform(["hate", "terrible", "bad"])))

[1. 1. 1.]


In [58]:
print(df["label"].value_counts())

label
1.0    101
0.0     12
Name: count, dtype: int64
